In [82]:
from __future__ import annotations

import operator
from typing import TypedDict, List, Annotated

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_openai import ChatOpenAI
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

In [83]:
class Task(BaseModel):
    id: str
    title: str
    brief: str = Field(..., description="What to cover")

In [84]:
class Plan(BaseModel):
    blog_title: str
    tasks: List[Task]

In [85]:
class State(TypedDict):
    topic: str
    plan: Plan
    sections: Annotated[List[str], operator.add]
    final: str

In [86]:
from langchain_google_genai import ChatGoogleGenerativeAI

import os
api_key = os.getenv('GEMINI_API_KEY')
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=api_key)

In [87]:
def orchestrator(state: State) -> dict:

    plan = llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(content="Create a blog of 3 - 4 sections with the following topic."),
            HumanMessage(content=f"Topic: {state['topic']}"),
        ]
    )

    return {"plan": plan}

In [88]:
def fanout(state: State):
    return [Send("worker", {"task": task, "plan": state["plan"]}) for task in state["plan"].tasks]

In [89]:
def workload(payload: dict) -> dict:
    task = payload["task"]
    topic = payload["plan"]
    plan = payload["plan"]

    blog_title = plan.blog_title

    section_md = llm.invoke(
        [
            SystemMessage(content=f"Write one clean markdown section."),
            HumanMessage(content=
                         (f"Blog: {blog_title}\n"
                          f"Topic: {topic}\n"
                          f"Section: {task.title}\n"
                          f"Brief: {task.brief}\n"
                          "Return only in markdown format.")),
        ]
    ).content.strip()

    return {"section": section_md}

In [90]:
from pathlib import Path

def reducer(state: State) -> dict:

    title = state["plan"].blog_title
    sections = "\n".join(state["sections"]).strip()

    final_md = f"# {title}\n\n{sections}\n"

    filename = title.lower().replace(" ", "_") + ".md"
    output_path = Path(filename)
    output_path.write_text(final_md, encoding="utf-8")

    return {"final": final_md} 

In [91]:
g = StateGraph(State)
g.add_node("orchestrator", orchestrator)
g.add_node("worker", workload)
g.add_node("reducer", reducer)


In [92]:
g.add_edge(START, "orchestrator")
g.add_conditional_edges("orchestrator", fanout, ["worker"])
g.add_edge("worker", "reducer")
g.add_edge("reducer", END)

app = g.compile()

In [93]:
out = app.invoke({"topic": "The Future of AI in Healthcare"})

In [94]:
print(out["plan"])

blog_title='The Future of AI in Healthcare: Revolutionizing Patient Care and Medical Innovation' tasks=[Task(id='section1', title='Introduction: A New Era for Medicine', brief='Introduce the transformative potential of Artificial Intelligence in the healthcare sector, moving beyond traditional methods to usher in an era of unprecedented innovation and efficiency. Highlight the growing challenges in healthcare that AI aims to address.'), Task(id='section2', title='Key Applications: Where AI is Making an Impact', brief='Explore specific areas where AI is already making significant strides or is poised to do so. This includes enhanced diagnostics and imaging analysis, accelerated drug discovery and development, personalized treatment plans, and operational efficiencies in hospitals and clinics.'), Task(id='section3', title='Navigating Challenges and Ethical Considerations', brief="Discuss the hurdles and ethical dilemmas associated with AI integration in healthcare. Cover topics such as d